# =============================================================================
# PART 2 — LOGISTIC REGRESSION FROM SCRATCH
# Full pipeline: EDA -> cleaning -> feature selection -> training -> evaluation
# =============================================================================

# 0. Imports 

In [34]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, log_loss,
    precision_recall_curve, auc
)
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 1. Load & inspect 

In [36]:
df = pd.read_csv('logistic_regression.csv')
print("Shape:", df.shape)
print(df.dtypes)
print(df.head())
print(df.describe(include='all').T)

Shape: (5000, 16)
book_id                     object
genre                       object
publisher_size              object
release_quarter             object
author_popularity_score    float64
marketing_budget_usd       float64
price_usd                  float64
page_count                   int64
advance_review_count         int64
average_rating             float64
social_media_mentions        int64
preorders                    int64
distribution_outlets         int64
is_series_book               int64
award_nomination             int64
best_seller                  int64
dtype: object
   book_id    genre publisher_size release_quarter  author_popularity_score  \
0  BK00001  Fantasy          Small              Q1                    27.83   
1  BK00002  Fiction          Small              Q1                    20.74   
2  BK00003  Fiction         Medium              Q2                    19.55   
3  BK00004  Fiction          Small              Q4                    44.02   
4  BK00005  F

# 2. Data quality checks 

In [37]:
# Missing values
print("\nMissing values:\n", df.isna().sum())
# Duplicates
print("\nDuplicate book_id:", df['book_id'].duplicated().sum())
print("Duplicate rows:", df.duplicated().sum())
# Target validity
print("Target values:", df['best_seller'].unique())
# Rating range
print("Rating range:", df['average_rating'].min(), df['average_rating'].max())
# Negative values in numeric columns
num_cols = ['author_popularity_score','marketing_budget_usd','price_usd','page_count',
            'advance_review_count','average_rating','social_media_mentions','preorders',
            'distribution_outlets']
print("Negative counts:", {c: (df[c] < 0).sum() for c in num_cols})
# Categorical values
for c in ['genre','publisher_size','release_quarter']:
    print(c, df[c].unique())


Missing values:
 book_id                    0
genre                      0
publisher_size             0
release_quarter            0
author_popularity_score    0
marketing_budget_usd       0
price_usd                  0
page_count                 0
advance_review_count       0
average_rating             0
social_media_mentions      0
preorders                  0
distribution_outlets       0
is_series_book             0
award_nomination           0
best_seller                0
dtype: int64

Duplicate book_id: 0
Duplicate rows: 0
Target values: [0 1]
Rating range: 2.05 5.0
Negative counts: {'author_popularity_score': np.int64(0), 'marketing_budget_usd': np.int64(0), 'price_usd': np.int64(0), 'page_count': np.int64(0), 'advance_review_count': np.int64(0), 'average_rating': np.int64(0), 'social_media_mentions': np.int64(0), 'preorders': np.int64(0), 'distribution_outlets': np.int64(0)}
genre ['Fantasy' 'Fiction' 'Nonfiction' 'Self_Help' 'Science_Fiction' 'Mystery'
 'Biography' 'Romance' '

# 3. Exploratory Data Analysis 

In [38]:
# 3.1 Target balance
print(df['best_seller'].value_counts(normalize=True))
fig, ax = plt.subplots()
df['best_seller'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('Target distribution (best_seller)')
ax.set_xlabel('best_seller'); ax.set_ylabel('Count')
plt.tight_layout(); plt.savefig('eda_target.png'); plt.close()

best_seller
1    0.519
0    0.481
Name: proportion, dtype: float64


In [39]:
# 3.2 Numeric distributions
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, col in zip(axes.ravel(), num_cols):
    df[col].hist(ax=ax, bins=40)
    ax.set_title(col)
plt.tight_layout(); plt.savefig('eda_numeric_hist.png'); plt.close()

In [40]:
# 3.3 Numeric vs target
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, col in zip(axes.ravel(), num_cols):
    sns.boxplot(x='best_seller', y=col, data=df, ax=ax)
plt.tight_layout(); plt.savefig('eda_numeric_box.png'); plt.close()

In [41]:
# 3.4 Categorical vs target
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ['genre','publisher_size','release_quarter']):
    rate = df.groupby(col)['best_seller'].mean().sort_values()
    rate.plot(kind='barh', ax=ax); ax.set_title(f'Bestseller rate by {col}')
plt.tight_layout(); plt.savefig('eda_categorical_rate.png'); plt.close()

In [42]:
# 3.5 Correlation
corr = df[num_cols + ['best_seller']].corr()
fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
plt.tight_layout(); plt.savefig('eda_corr.png'); plt.close()

In [43]:
# 3.6 Outlier counts (IQR)
out_counts = {}
for col in num_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    out_counts[col] = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
print("Outliers (IQR):", out_counts)

Outliers (IQR): {'author_popularity_score': np.int64(0), 'marketing_budget_usd': np.int64(273), 'price_usd': np.int64(28), 'page_count': np.int64(19), 'advance_review_count': np.int64(32), 'average_rating': np.int64(32), 'social_media_mentions': np.int64(380), 'preorders': np.int64(380), 'distribution_outlets': np.int64(17)}


# 4. Data cleaning 

In [47]:
df_clean = df.copy()
df_clean = df_clean.drop(columns=['book_id'])
# (no missing/duplicates/invalid in this synthetic dataset, but retain safety)
df_clean = df_clean.drop_duplicates()
df_clean = df_clean[(df_clean['average_rating'].between(0,5)) & (df_clean['best_seller'].isin([0,1]))]
print("Clean shape:", df_clean.shape)

Clean shape: (5000, 15)


# 5. Split BEFORE encoding & scaling (avoid leakage) 

In [48]:
X_raw = df_clean.drop(columns=['best_seller'])
y = df_clean['best_seller'].astype(int).values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train_raw.shape, "Test:", X_test_raw.shape)

Train: (4000, 14) Test: (1000, 14)


# 6. Encoding 

In [49]:

cat_cols = ['genre','publisher_size','release_quarter']
X_train_enc = pd.get_dummies(X_train_raw, columns=cat_cols, drop_first=True)
X_test_enc = pd.get_dummies(X_test_raw, columns=cat_cols, drop_first=True)
# Align columns (test may have missing categories)
X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join='left', axis=1, fill_value=0)
print("Encoded feature count:", X_train_enc.shape[1])

Encoded feature count: 25


# 7. Feature selection (statsmodels backward elimination on TRAIN only) 

In [50]:
def backward_elimination(X, y, threshold=0.05, verbose=True):
    X = sm.add_constant(X)
    feature_names = list(X.columns)
    while len(feature_names) > 1:  # keep const
        model = sm.Logit(y, X[feature_names]).fit(disp=0)
        pvals = model.pvalues
        # exclude const from removal consideration
        drop_candidates = pvals.drop('const') if 'const' in pvals.index else pvals
        max_p = drop_candidates.max()
        if max_p > threshold:
            drop_var = drop_candidates.idxmax()
            feature_names.remove(drop_var)
            if verbose: print(f"Dropped {drop_var} (p={max_p:.4f})")
        else:
            break
    return feature_names, model

selected_features_with_const, final_model = backward_elimination(
    X_train_enc.astype(float), y_train, threshold=0.05
)
selected_features = [f for f in selected_features_with_const if f != 'const']
print("\nSelected features:", selected_features)

# VIF on selected numeric features (recompute)
X_train_sel = X_train_enc[selected_features].astype(float)
vif_data = pd.DataFrame({
    'feature': X_train_sel.columns,
    'VIF': [variance_inflation_factor(X_train_sel.values, i) for i in range(X_train_sel.shape[1])]
})
print("\nVIF:\n", vif_data.sort_values('VIF', ascending=False))

Dropped page_count (p=0.9515)
Dropped genre_Self_Help (p=0.7417)
Dropped preorders (p=0.7016)
Dropped genre_Nonfiction (p=0.5049)
Dropped genre_Mystery (p=0.6462)
Dropped release_quarter_Q2 (p=0.4069)
Dropped release_quarter_Q3 (p=0.3931)
Dropped genre_Children (p=0.1988)
Dropped genre_Fiction (p=0.2504)
Dropped genre_Science_Fiction (p=0.3633)
Dropped social_media_mentions (p=0.0994)

Selected features: ['author_popularity_score', 'marketing_budget_usd', 'price_usd', 'advance_review_count', 'average_rating', 'distribution_outlets', 'is_series_book', 'award_nomination', 'genre_Fantasy', 'genre_Romance', 'genre_Thriller', 'publisher_size_Medium', 'publisher_size_Small', 'release_quarter_Q4']

VIF:
                     feature        VIF
3      advance_review_count  36.760313
4            average_rating  35.562944
2                 price_usd  12.839423
5      distribution_outlets   7.299206
0   author_popularity_score   6.484454
12     publisher_size_Small   3.966183
11    publisher_size

# 8. Scaling (fit on train only) 

In [51]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled = scaler.transform(X_test_enc[selected_features].astype(float))

# 9. From-scratch logistic regression 

In [53]:
def sigmoid(z):
    return np.where(z >= 0, 1.0/(1.0 + np.exp(-z)), np.exp(z)/(1.0 + np.exp(z)))

def compute_loss(y, y_hat, w, b, lam=0.0):
    m = len(y)
    eps = 1e-15
    y_hat = np.clip(y_hat, eps, 1-eps)
    loss = -np.mean(y*np.log(y_hat) + (1-y)*np.log(1-y_hat))
    loss += lam/(2*m) * np.sum(w**2)
    return loss

def compute_gradients(X, y, y_hat, w, b, lam=0.0):
    m = len(y)
    err = y_hat - y
    dw = (X.T @ err)/m + (lam/m)*w
    db = np.mean(err)
    return dw, db

def train_logistic(X, y, lr=0.01, n_iter=10000, lam=0.0, tol=1e-6, verbose=False):
    m, n = X.shape
    w = np.zeros(n); b = 0.0
    losses = []
    prev = np.inf
    for it in range(n_iter):
        y_hat = sigmoid(X @ w + b)
        loss = compute_loss(y, y_hat, w, b, lam)
        losses.append(loss)
        dw, db = compute_gradients(X, y, y_hat, w, b, lam)
        w -= lr*dw; b -= lr*db
        if abs(prev - loss) < tol:
            if verbose: print(f"Converged @ iter {it}, loss={loss:.6f}")
            break
        prev = loss
    return w, b, losses

def predict_proba(X, w, b):
    return sigmoid(X @ w + b)

def predict(X, w, b, threshold=0.5):
    return (predict_proba(X, w, b) >= threshold).astype(int)

# 10. Cross-validated hyperparameter tuning 

In [54]:
def cv_score(X, y, lr, n_iter, lam, folds=5):
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for tr, va in skf.split(X, y):
        X_tr, X_va = X[tr], X[va]
        y_tr, y_va = y[tr], y[va]
        sc = StandardScaler().fit(X_tr)
        X_tr_s, X_va_s = sc.transform(X_tr), sc.transform(X_va)
        w, b, L = train_logistic(X_tr_s, y_tr, lr, n_iter, lam)
        p_va = predict_proba(X_va_s, w, b)
        p_tr = predict_proba(X_tr_s, w, b)
        rows.append({
            'logloss': log_loss(y_va, p_va),
            'roc_auc': roc_auc_score(y_va, p_va),
            'f1': f1_score(y_va, (p_va>=0.5).astype(int)),
            'acc': accuracy_score(y_va, (p_va>=0.5).astype(int)),
            'iters': len(L),
            'train_ll': log_loss(y_tr, p_tr),
        })
    return pd.DataFrame(rows)

# For CV here, X is the UNSCALED selected features (scaling done per fold)
X_train_sel_np = X_train_sel.values

grid = []
for lr in [0.001, 0.01, 0.1]:
    for n_iter in [1000, 5000, 10000]:
        for lam in [0.0, 0.01, 0.1, 1.0]:
            res = cv_score(X_train_sel_np, y_train, lr, n_iter, lam)
            grid.append({
                'lr': lr, 'n_iter': n_iter, 'lam': lam,
                'cv_logloss': res['logloss'].mean(),
                'cv_roc_auc': res['roc_auc'].mean(),
                'cv_f1': res['f1'].mean(),
                'cv_acc': res['acc'].mean(),
                'cv_train_ll': res['train_ll'].mean(),
            })
grid_df = pd.DataFrame(grid).sort_values('cv_logloss')
print("\nTop 10 hyperparameter combinations:\n", grid_df.head(10))

best = grid_df.iloc[0]
print("\nBest hyperparameters:", best.to_dict())


Top 10 hyperparameter combinations:
       lr  n_iter   lam  cv_logloss  cv_roc_auc     cv_f1   cv_acc  cv_train_ll
20  0.01   10000  0.00    0.619125    0.712894  0.671882  0.65625     0.615076
16  0.01    5000  0.00    0.619125    0.712894  0.671882  0.65625     0.615076
17  0.01    5000  0.01    0.619125    0.712894  0.671882  0.65625     0.615076
21  0.01   10000  0.01    0.619125    0.712894  0.671882  0.65625     0.615076
22  0.01   10000  0.10    0.619126    0.712897  0.671882  0.65625     0.615078
18  0.01    5000  0.10    0.619126    0.712897  0.671882  0.65625     0.615078
23  0.01   10000  1.00    0.619129    0.712866  0.671727  0.65600     0.615090
19  0.01    5000  1.00    0.619129    0.712866  0.671727  0.65600     0.615090
31  0.10    5000  1.00    0.619267    0.712927  0.673341  0.65725     0.614690
27  0.10    1000  1.00    0.619267    0.712927  0.673341  0.65725     0.614690

Best hyperparameters: {'lr': 0.01, 'n_iter': 10000.0, 'lam': 0.0, 'cv_logloss': 0.6191248137

# 11. Refit on full training set with best hyperparameters 

In [55]:
w_final, b_final, losses_final = train_logistic(
    X_train_scaled, y_train, lr=best['lr'], n_iter=int(best['n_iter']), lam=best['lam']
)

# Loss curve
fig, ax = plt.subplots()
ax.plot(losses_final); ax.set_xlabel('Iteration'); ax.set_ylabel('Loss')
ax.set_title('Training loss curve')
plt.tight_layout(); plt.savefig('loss_curve.png'); plt.close()

# 12. Evaluation on test set

In [56]:
p_test = predict_proba(X_test_scaled, w_final, b_final)
y_pred = (p_test >= 0.5).astype(int)

metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, p_test),
    'Log-loss': log_loss(y_test, p_test),
}
print("\nTest metrics:")
for k, v in metrics.items(): print(f"  {k}: {v:.4f}")


Test metrics:
  Accuracy: 0.6420
  Precision: 0.6522
  Recall: 0.6647
  F1: 0.6584
  ROC-AUC: 0.7111
  Log-loss: 0.6188


# Confusion matrix

In [57]:
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion matrix:\n", cm)
fig, ax = plt.subplots()
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not BS','BS'], yticklabels=['Not BS','BS'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (test)')
plt.tight_layout(); plt.savefig('confusion_matrix.png'); plt.close()


Confusion matrix:
 [[297 184]
 [174 345]]


# ROC curve

In [58]:
fpr, tpr, _ = roc_curve(y_test, p_test)
fig, ax = plt.subplots()
ax.plot(fpr, tpr, label=f'AUC = {metrics["ROC-AUC"]:.3f}')
ax.plot([0,1],[0,1],'k--')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC curve')
ax.legend(); plt.tight_layout(); plt.savefig('roc_curve.png'); plt.close()

# Precision/Recall vs threshold

In [59]:
thresholds = np.linspace(0.1, 0.9, 17)
prec_list, rec_list, f1_list = [], [], []
for t in thresholds:
    yp = (p_test >= t).astype(int)
    prec_list.append(precision_score(y_test, yp, zero_division=0))
    rec_list.append(recall_score(y_test, yp, zero_division=0))
    f1_list.append(f1_score(y_test, yp, zero_division=0))

fig, ax = plt.subplots()
ax.plot(thresholds, prec_list, label='Precision')
ax.plot(thresholds, rec_list, label='Recall')
ax.plot(thresholds, f1_list, label='F1')
ax.axvline(0.5, color='k', linestyle='--', alpha=0.5, label='Threshold 0.5')
ax.set_xlabel('Threshold'); ax.legend(); ax.set_title('Metrics vs threshold')
plt.tight_layout(); plt.savefig('threshold_analysis.png'); plt.close()

# 13. Coefficient interpretation (odds ratios) 

In [60]:
coef_df = pd.DataFrame({
    'Feature': selected_features,
    'Coefficient': w_final,
    'Odds Ratio': np.exp(w_final)
}).sort_values('Coefficient', key=abs, ascending=False)
print("\nCoefficient interpretation:\n", coef_df)


Coefficient interpretation:
                     Feature  Coefficient  Odds Ratio
0   author_popularity_score     0.484126    1.622756
1      marketing_budget_usd     0.253078    1.287983
4            average_rating     0.247083    1.280285
5      distribution_outlets     0.242138    1.273971
12     publisher_size_Small    -0.214911    0.806613
7          award_nomination     0.182830    1.200611
3      advance_review_count     0.182070    1.199698
8             genre_Fantasy     0.162337    1.176256
6            is_series_book     0.134246    1.143674
13       release_quarter_Q4     0.118543    1.125855
2                 price_usd    -0.093796    0.910468
11    publisher_size_Medium    -0.083553    0.919842
9             genre_Romance     0.082680    1.086194
10           genre_Thriller     0.066047    1.068277


# 14. Baseline comparison (majority class) 

In [61]:
baseline = max(np.mean(y_test), 1-np.mean(y_test))
print(f"\nMajority-class baseline accuracy: {baseline:.4f}")


Majority-class baseline accuracy: 0.5190


# 15. Library benchmark (optional parity check) 

In [62]:
from sklearn.linear_model import LogisticRegression
bench = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
bench.fit(X_train_scaled, y_train)
p_bench = bench.predict_proba(X_test_scaled)[:,1]
print(f"\nBenchmark (sklearn) ROC-AUC: {roc_auc_score(y_test, p_bench):.4f}")
print(f"Benchmark (sklearn) Log-loss: {log_loss(y_test, p_bench):.4f}")


Benchmark (sklearn) ROC-AUC: 0.7101
Benchmark (sklearn) Log-loss: 0.6194
